# PySpark DataFrame Data Engineering Project — Pharma Analytics

Business-oriented PySpark DataFrame project using two related input files.

**Input files:** `drugs.csv`, `drug_orders.csv`

**Questions:** 30

Topics include DataFrame creation, schema, select, filter, withColumn, when, cast, groupBy, agg, joins, orderBy, window functions, null handling, duplicates, date functions, pivot, union-style reporting, and business KPIs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkDataFrameProject").getOrCreate()

df1 = spark.read.option("header", True).option("inferSchema", True).csv("drugs.csv")
df2 = spark.read.option("header", True).option("inferSchema", True).csv("drug_orders.csv")

df1.show(5, truncate=False)
df2.show(5, truncate=False)


## Q1. Inspect Schemas

Display both DataFrame schemas and row counts.

In [ ]:
df1.printSchema()
df2.printSchema()
print(df1.count(), df2.count())

## Q2. Select Business Columns

Select drug_id and category from the first DataFrame.

In [ ]:
df1.select('drug_id','category').show(20, truncate=False)

## Q3. Filter Records

Filter the second DataFrame where order_value is greater than its average.

In [ ]:
avg_value = df2.select(avg('order_value').alias('avg')).first()['avg']
df2.filter(col('order_value') > avg_value).show(20, truncate=False)

## Q4. Create Derived Column

Create a new column based on order_value using a High/Normal classification.

In [ ]:
df2.withColumn('value_category', when(col('order_value') >= avg('order_value').over(Window.partitionBy()), 'High').otherwise('Normal')).show(20, truncate=False)

## Q5. Null Analysis

Show null counts for every column in both DataFrames.

In [ ]:
df1.select([sum(col(c).isNull().cast('int')).alias(c) for c in df1.columns]).show()
df2.select([sum(col(c).isNull().cast('int')).alias(c) for c in df2.columns]).show()

## Q6. Handle Null Values

Fill null values in category and hospital with 'Unknown'.

In [ ]:
df1.fillna({'category': 'Unknown'}).show(5)
df2.fillna({'hospital': 'Unknown'}).show(5)

## Q7. Remove Duplicates

Remove duplicate records and compare row counts.

In [ ]:
print('Before:', df1.count(), df2.count())
print('After:', df1.dropDuplicates().count(), df2.dropDuplicates().count())

## Q8. Inner Join

Join the two DataFrames using drug_id and drug_id.

In [ ]:
joined = df1.join(df2, df1['drug_id'] == df2['drug_id'], 'inner')
joined.show(20, truncate=False)

## Q9. Left Join

Perform a left join and identify unmatched records.

In [ ]:
left = df1.join(df2, df1['drug_id'] == df2['drug_id'], 'left')
left.show(20, truncate=False)

## Q10. Group By Category

Calculate count and average order_value by hospital.

In [ ]:
df2.groupBy('hospital').agg(count('*').alias('records'), avg('order_value').alias('avg_value')).orderBy(desc('avg_value')).show()

## Q11. Total by Category

Calculate total order_value by hospital.

In [ ]:
df2.groupBy('hospital').agg(sum('order_value').alias('total_value')).orderBy(desc('total_value')).show()

## Q12. Maximum by Category

Find maximum order_value in each hospital.

In [ ]:
df2.groupBy('hospital').agg(max('order_value').alias('max_value')).show()

## Q13. Minimum by Category

Find minimum order_value in each hospital.

In [ ]:
df2.groupBy('hospital').agg(min('order_value').alias('min_value')).show()

## Q14. Having Condition

Show hospital groups whose total order_value is greater than the overall average order_value.

In [ ]:
overall = df2.select(avg('order_value').alias('v')).first()['v']
df2.groupBy('hospital').agg(sum('order_value').alias('total')).filter(col('total') > overall).show()

## Q15. Top 10 Records

Find the top 10 records by order_value.

In [ ]:
df2.orderBy(desc('order_value')).limit(10).show(truncate=False)

## Q16. Bottom 10 Records

Find the bottom 10 records by order_value.

In [ ]:
df2.orderBy(asc('order_value')).limit(10).show(truncate=False)

## Q17. Window Rank

Rank records by order_value within each hospital.

In [ ]:
w = Window.partitionBy('hospital').orderBy(desc('order_value'))
df2.withColumn('rank', dense_rank().over(w)).show(30, truncate=False)

## Q18. Running Total

Calculate a running total of order_value within each hospital.

In [ ]:
w = Window.partitionBy('hospital').orderBy('order_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
df2.withColumn('running_total', sum('order_value').over(w)).show(30, truncate=False)

## Q19. Previous Value

Use lag to compare each record's order_value with the previous record in its hospital.

In [ ]:
w = Window.partitionBy('hospital').orderBy('order_date')
df2.withColumn('previous_value', lag('order_value').over(w)).show(30, truncate=False)

## Q20. Percentage Contribution

Calculate each hospital's contribution to total order_value.

In [ ]:
tot = df2.select(sum('order_value').alias('t')).first()['t']
df2.groupBy('hospital').agg(sum('order_value').alias('total')).withColumn('pct', col('total')/lit(tot)*100).orderBy(desc('pct')).show()

## Q21. Date Extraction

Extract year and month from order_date.

In [ ]:
df2.withColumn('year', year('order_date')).withColumn('month', month('order_date')).show(20, truncate=False)

## Q22. Monthly KPI

Calculate total and average order_value by month.

In [ ]:
df2.withColumn('month', date_format('order_date','yyyy-MM')).groupBy('month').agg(sum('order_value').alias('total'), avg('order_value').alias('average')).orderBy('month').show()

## Q23. Pivot Report

Create a pivot report with hospital as rows and year of order_date as columns.

In [ ]:
df2.withColumn('year', year('order_date')).groupBy('hospital').pivot('year').agg(sum('order_value')).show()

## Q24. Conditional KPI

Count records where order_value is above the category average.

In [ ]:
w=Window.partitionBy('hospital')
df2.withColumn('cat_avg',avg('order_value').over(w)).filter(col('order_value')>col('cat_avg')).groupBy('hospital').count().show()

## Q25. Join Aggregation

Join the DataFrames and calculate total order_value by category.

In [ ]:
df1.join(df2, df1['drug_id']==df2['drug_id']).groupBy(df1['category']).agg(sum(df2['order_value']).alias('total')).orderBy(desc('total')).show()

## Q26. Join Ranking

After joining, rank category by total order_value.

In [ ]:
r=df1.join(df2,df1['drug_id']==df2['drug_id']).groupBy(df1['category']).agg(sum(df2['order_value']).alias('total'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).show()

## Q27. Business Segmentation

Create Low, Medium and High segments for order_value.

In [ ]:
df2.withColumn('segment',when(col('order_value')<1000,'Low').when(col('order_value')<5000,'Medium').otherwise('High')).groupBy('segment').count().show()

## Q28. Data Quality Check

Count invalid negative order_value values.

In [ ]:
df2.filter(col('order_value')<0).count()

## Q29. Final Business Report

Create a final report by hospital with count, total, average, minimum, maximum and rank.

In [ ]:
r=df2.groupBy('hospital').agg(count('*').alias('records'),sum('order_value').alias('total'),avg('order_value').alias('average'),min('order_value').alias('minimum'),max('order_value').alias('maximum'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).orderBy('rank').show(truncate=False)

## End of Project